# DML in 5 Minutes

**Goal:** Run a complete Double/Debiased Machine Learning estimation in under 2 minutes of runtime.

**What you will learn:** How to estimate a causal effect with many controls using `doubleml`.

**Scenario:** Estimate the effect of a policy change on a market outcome, controlling for 50 confounders.

---

## Setup

Install the only library beyond standard ML tools:

In [ ]:
!pip install -q doubleml

## Quick Win: Simulate Data with Known Truth

We create a dataset where the true causal effect is exactly 0.5.
There are 50 confounders with nonlinear effects — OLS would be biased.

In [ ]:
import numpy as np
import pandas as pd
from doubleml import DoubleMLData, DoubleMLPLR
from sklearn.ensemble import GradientBoostingRegressor

np.random.seed(42)
n, p = 2000, 50
true_effect = 0.5

X = np.random.randn(n, p)
D = 0.5 * np.sin(X[:, 0]) + 0.3 * X[:, 1]**2 + np.random.randn(n) * 0.5
Y = true_effect * D + np.exp(0.2 * X[:, 0]) + 0.3 * X[:, 2] + np.random.randn(n) * 0.3

col_names = [f'X{i}' for i in range(p)]
df = pd.DataFrame(X, columns=col_names)
df['outcome'] = Y
df['treatment'] = D

print(f'Data: {n} observations, {p} controls')
print(f'True causal effect: {true_effect}')

## Run DML in 4 Lines

Create the data object, choose ML models, fit, and read the result.

In [ ]:
# 1. Wrap data
dml_data = DoubleMLData(df, y_col='outcome', d_cols='treatment', x_cols=col_names)

# 2. Choose ML models for nuisance functions
ml_l = GradientBoostingRegressor(200, max_depth=5, random_state=42)
ml_m = GradientBoostingRegressor(200, max_depth=5, random_state=42)

# 3. Fit DML
dml = DoubleMLPLR(dml_data, ml_l=ml_l, ml_m=ml_m, n_folds=5)
dml.fit()

# 4. Results
print(dml.summary)
print(f'\nTrue effect:  {true_effect}')
print(f'DML estimate: {dml.coef[0]:.4f}')
print(f'95% CI:       [{dml.confint().iloc[0, 0]:.4f}, {dml.confint().iloc[0, 1]:.4f}]')

## What Just Happened?

DML did three things automatically:
1. **Residualised** Y and D using gradient boosting (removed confounding)
2. **Cross-fitted** to avoid overfitting bias (5-fold out-of-sample predictions)
3. **Computed** the treatment effect from residuals with valid standard errors

The result: a causal estimate close to 0.5, with a tight confidence interval, despite 50 nonlinear confounders.

## What is Next?

**Continue learning:**
- Module 00: Why naive regression fails for causal inference
- Module 02: The orthogonalisation trick (what DML does internally)
- Module 05: Full PLR pipeline with model comparison
- Module 08: Heterogeneous treatment effects (CATE)

**Production use:**
- `templates/dml_pipeline.py` — Production pipeline scaffold
- `templates/cate_analysis.py` — CATE estimation template
- `recipes/` — Copy-paste code patterns

**Key insight:** DML = ML for prediction + econometrics for inference.
It handles high-dimensional confounding that OLS cannot.